In [10]:
import arcpy
import os
import csv
import math

# =====================================================
# USER INPUTS
# =====================================================
ROW_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\nulls"
OUT_GDB = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb"
OUT_PREFIX = "CONUS_ROW_MRR"

ROW_ID_FIELD = "ROW_ID"
WIDTH_FIELD = "ROW_Width_Meters"
CSV_OUTPUT = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_row_widths_MRR.csv"

arcpy.env.overwriteOutput = True
arcpy.env.workspace = OUT_GDB

# =====================================================
# STEP 1: MINIMUM ROTATED RECTANGLES
# =====================================================
#mrr_fc = os.path.join(OUT_GDB, f"{OUT_PREFIX}_MBR")

mrr_fc = os.path.join(OUT_GDB, OUT_PREFIX)

arcpy.management.MinimumBoundingGeometry(
    ROW_FC,
    mrr_fc,
    "RECTANGLE_BY_WIDTH"
)

# =====================================================
# STEP 2: COMPUTE WIDTH FROM RECTANGLE GEOMETRY
# =====================================================
width_dict = {}

with arcpy.da.SearchCursor(mrr_fc, [ROW_ID_FIELD, "SHAPE@"]) as cursor:
    for row_id, geom in cursor:
        if geom is None:
            continue

        # Rectangle has one part with 5 points (last = first)
        pts = geom.getPart(0)

        edges = []
        for i in range(4):
            p1 = pts[i]
            p2 = pts[i + 1]
            length = math.hypot(p2.X - p1.X, p2.Y - p1.Y)
            edges.append(length)

        # Two unique edge lengths → width is the smaller
        width_dict[row_id] = min(edges)

# =====================================================
# STEP 3: WRITE WIDTH BACK TO ROW POLYGONS
# =====================================================
row_fields = [f.name for f in arcpy.ListFields(ROW_FC)]
if WIDTH_FIELD not in row_fields:
    arcpy.management.AddField(ROW_FC, WIDTH_FIELD, "DOUBLE")

with arcpy.da.UpdateCursor(ROW_FC, [ROW_ID_FIELD, WIDTH_FIELD]) as cursor:
    for row_id, _ in cursor:
        cursor.updateRow([row_id, width_dict.get(row_id)])

# =====================================================
# STEP 4: OPTIONAL CSV OUTPUT
# =====================================================
if CSV_OUTPUT:
    with open(CSV_OUTPUT, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([ROW_ID_FIELD, WIDTH_FIELD])
        for k, v in width_dict.items():
            writer.writerow([k, v])

print("ROW width calculation complete — geometry-verified.")

ROW width calculation complete — geometry-verified.


In [11]:
import arcpy
import csv

# ============================
# USER INPUTS
# ============================
ROW_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"
ROW_ID_FIELD = "ROW_ID"
WIDTH_FIELD = "ROW_Width_Meters"

# CSV with calculated widths
CSV_FILE = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_row_widths_MRR.csv"
# CSV format: ROW_ID_OG, approx_length_meters
# e.g. 1, 35.2

# ============================
# LOAD CSV INTO DICTIONARY
# ============================
width_dict = {}
with open(CSV_FILE, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = row[ROW_ID_FIELD]
        try:
            val = float(row[WIDTH_FIELD])
            width_dict[key] = val
        except (ValueError, KeyError):
            # Skip rows with missing/invalid data
            continue

# ============================
# UPDATE ROW_FC ONLY WHERE NULL
# ============================
with arcpy.da.UpdateCursor(ROW_FC, [ROW_ID_FIELD, WIDTH_FIELD]) as cursor:
    for row_id, width in cursor:
        # Only update if currently NULL
        if width is None:
            new_width = width_dict.get(str(row_id))  # CSV keys are read as strings
            if new_width is not None:
                cursor.updateRow([row_id, new_width])

print("Null approx_length_meters values updated successfully.")

Null approx_length_meters values updated successfully.


In [12]:
import arcpy

# =====================================================
# USER INPUT
# =====================================================
ROW_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

METERS_FIELD = "ROW_Width_Meters"
FEET_FIELD = "ROW_Width_Feet"

# Meters to US survey feet
METERS_TO_US_FEET = 3.28083333333333

# =====================================================
# CHECK FIELDS
# =====================================================
fields = [f.name for f in arcpy.ListFields(ROW_FC)]

if METERS_FIELD not in fields:
    raise ValueError(f"Field not found: {METERS_FIELD}")

if FEET_FIELD not in fields:
    raise ValueError(f"Field not found: {FEET_FIELD}")

# =====================================================
# UPDATE NULL FEET VALUES
# =====================================================
updated_count = 0
skipped_count = 0

with arcpy.da.UpdateCursor(
    ROW_FC,
    [METERS_FIELD, FEET_FIELD]
) as cursor:

    for meters, feet in cursor:

        # Only process records where feet is NULL
        if feet is None:

            # Make sure a meter value exists
            if meters is not None:
                feet_value = meters * METERS_TO_US_FEET

                cursor.updateRow([
                    meters,
                    feet_value
                ])

                updated_count += 1

            else:
                skipped_count += 1

# =====================================================
# SUMMARY
# =====================================================
print("=" * 60)
print("ROW WIDTH FEET UPDATE COMPLETE")
print("=" * 60)
print(f"Records updated: {updated_count}")
print(f"Records skipped (both values NULL): {skipped_count}")
print(f"Conversion factor: {METERS_TO_US_FEET} US survey feet per meter")
print("=" * 60)

ROW WIDTH FEET UPDATE COMPLETE
Records updated: 562
Records skipped (both values NULL): 0
Conversion factor: 3.28083333333333 US survey feet per meter
